# Train Version 2 Outcome Model

This notebook trains the simplified Version 2 architecture.

Active trained model: CatBoostClassifier for match outcome only.

Goal prediction is handled later by the rating-based Poisson model in the prediction script.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "version_2_ml" / "scripts").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "version_2_ml":
    PROJECT_ROOT = cwd.parent
elif cwd.name == "notebooks" and cwd.parent.name == "version_2_ml":
    PROJECT_ROOT = cwd.parents[1]
else:
    PROJECT_ROOT = cwd

sys.path.append(str(PROJECT_ROOT / "version_2_ml" / "scripts"))

from train_version_2_models import (
    DEFAULT_RESULTS_PATHS,
    DEFAULT_FIFA_PATHS,
    find_existing_path,
    load_results,
    load_latest_fifa,
    build_training_dataset,
    train_models,
    save_training_outputs,
)


## Load Historical Results

In [ ]:
results_path = find_existing_path(DEFAULT_RESULTS_PATHS, "historical results")
fifa_path = find_existing_path(DEFAULT_FIFA_PATHS, "latest FIFA rankings", required=False)

results = load_results(results_path)
latest_fifa = load_latest_fifa(fifa_path)

results.head()


## Build Leakage-Aware Training Dataset

Keep `allow_latest_rating_features=False` unless you deliberately accept the leakage risk of using current FIFA rankings for old matches.

In [ ]:
allow_latest_rating_features = False

training_dataset = build_training_dataset(
    results,
    fifa_table=latest_fifa,
    allow_latest_rating_features=allow_latest_rating_features,
)

training_dataset.head()


## Train CatBoost Outcome Classifier

In [ ]:
training_result = train_models(
    training_dataset,
    train_fraction=0.8,
    random_seed=42,
    iterations=500,
)

training_result["metrics"]


## Feature Importance

In [ ]:
training_result["feature_importance"].head(20)


## Save Training Outputs

In [ ]:
save_training_outputs(
    training_dataset,
    training_result,
    allow_latest=allow_latest_rating_features,
)

print("Saved active Version 2 outcome model.")
